In [1]:
# 1. Importar librerías
from pathlib import Path
import time
import joblib
import polars as pl
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

# 2. Definir rutas
RUTA_PROYECTO = Path.cwd().parent
RUTA_DATASET = RUTA_PROYECTO / "data" / "dataset_proyecto.csv"
RUTA_PREPROCESADOR = RUTA_PROYECTO / "models" / "preprocessor.joblib"
RUTA_MODELOS = RUTA_PROYECTO / "models"

# 3. Cargar el dataset y el preprocesador
df = pl.read_csv(RUTA_DATASET)
preprocessor = joblib.load(RUTA_PREPROCESADOR)

print("Dataset cargado:", df.shape)
print("Preprocesador cargado correctamente.")

Dataset cargado: (6362620, 11)
Preprocesador cargado correctamente.


In [2]:
# 4. División temporal
df_train = df.filter(pl.col("step") <= 400)
df_prod = df.filter(pl.col("step") >= 401)

# 5. Definir variable objetivo y predictores
TARGET = "isFraud"

FEATURES = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# 6. Separar X e y
X_train = df_train.select(FEATURES)
y_train = df_train[TARGET]

X_prod = df_prod.select(FEATURES)
y_prod = df_prod[TARGET]

print("Entrenamiento:", X_train.shape, y_train.shape)
print("Producción:", X_prod.shape, y_prod.shape)

Entrenamiento: (5787030, 7) (5787030,)
Producción: (575590, 7) (575590,)


In [3]:
# 7. Convertir los predictores a Pandas
X_train_pd = pd.DataFrame(
    X_train.to_dict(as_series=False)
)

# 8. Aplicar el preprocesador guardado
X_train_transformed = preprocessor.transform(X_train_pd)

# 9. Convertir las etiquetas a NumPy
y_train_np = y_train.to_numpy()

# 10. Comprobar dimensiones
print("X_train:", X_train_transformed.shape)
print("y_train:", y_train_np.shape)
print("Tipo de matriz:", type(X_train_transformed))

X_train: (5787030, 11)
y_train: (5787030,)
Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>


In [4]:
# 11. Calcular el peso de las clases
conteo_clases = y_train.value_counts()

legitimos = int(
    conteo_clases.filter(pl.col("isFraud") == 0)["count"][0]
)

fraudes = int(
    conteo_clases.filter(pl.col("isFraud") == 1)["count"][0]
)

scale_pos_weight = legitimos / fraudes

print(f"Transacciones legítimas: {legitimos:,}")
print(f"Transacciones fraudulentas: {fraudes:,}")
print(f"Peso de la clase positiva: {scale_pos_weight:.2f}")

Transacciones legítimas: 5,782,553
Transacciones fraudulentas: 4,477
Peso de la clase positiva: 1291.61


In [5]:
# 12. Configurar XGBoost
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=4,
    tree_method="hist"
)

# 13. Entrenar con todos los datos de entrenamiento
inicio = time.time()

modelo_xgb.fit(
    X_train_transformed,
    y_train_np
)

tiempo_entrenamiento = (time.time() - inicio) / 60

print("Entrenamiento finalizado correctamente.")
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.2f} minutos")

Entrenamiento finalizado correctamente.
Tiempo de entrenamiento: 0.93 minutos


In [6]:
# 14. Guardar el modelo entrenado
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

RUTA_MODELO = RUTA_MODELOS / "xgboost_fraude.joblib"

joblib.dump(modelo_xgb, RUTA_MODELO)

print("Modelo guardado correctamente.")
print("Ruta:", RUTA_MODELO.resolve())

Modelo guardado correctamente.
Ruta: D:\ML_III\models\xgboost_fraude.joblib


In [7]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score
import xgboost as xgb
import numpy as np

print("Librerías para optimización cargadas correctamente")
print("Versión de XGBoost:", xgb.__version__)

Librerías para optimización cargadas correctamente
Versión de XGBoost: 3.2.0


In [8]:
# 1. Tomar una muestra reproducible del entrenamiento
muestra_opt = df_train.sample(n=200_000, seed=42)

# 2. Separar predictores y variable objetivo
X_opt = muestra_opt.select(FEATURES)
y_opt = muestra_opt["isFraud"].to_numpy()

# 3. Transformar usando el preprocesador ya ajustado
X_opt_pd = pd.DataFrame(X_opt.to_dict(as_series=False))
X_opt_transformed = preprocessor.transform(X_opt_pd)

# 4. Comprobar dimensiones y clases
print("Dimensiones de X:", X_opt_transformed.shape)
print("Dimensiones de y:", y_opt.shape)
print("Fraudes en la muestra:", int(y_opt.sum()))
print("Legítimas en la muestra:", int((y_opt == 0).sum()))

Dimensiones de X: (200000, 11)
Dimensiones de y: (200000,)
Fraudes en la muestra: 182
Legítimas en la muestra: 199818


In [9]:
# 1. Calcular el peso de clases en la muestra
legitimos_opt = int((y_opt == 0).sum())
fraudes_opt = int((y_opt == 1).sum())
peso_opt = legitimos_opt / fraudes_opt

# 2. Configurar el modelo base
modelo_busqueda = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=peso_opt,
    tree_method="hist",
    random_state=42,
    n_jobs=4
)

# 3. Definir el espacio de búsqueda
parametros = {
    "n_estimators": [100, 150, 200],
    "max_depth": [3, 5, 6],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# 4. Búsqueda aleatoria de 6 combinaciones
busqueda = RandomizedSearchCV(
    estimator=modelo_busqueda,
    param_distributions=parametros,
    n_iter=6,
    scoring="average_precision",
    cv=StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ),
    random_state=42,
    n_jobs=1,
    verbose=1,
    refit=True
)

# 5. Ejecutar la búsqueda
import time

inicio_opt = time.time()
busqueda.fit(X_opt_transformed, y_opt)
tiempo_opt = (time.time() - inicio_opt) / 60

print("\nOptimización finalizada")
print(f"Tiempo: {tiempo_opt:.2f} minutos")
print("Mejor Average Precision:", round(busqueda.best_score_, 4))
print("Mejores parámetros:")
print(busqueda.best_params_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Optimización finalizada
Tiempo: 1.03 minutos
Mejor Average Precision: 0.8283
Mejores parámetros:
{'subsample': 1.0, 'n_estimators': 150, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [10]:
import mlflow
from pathlib import Path

RUTA_PROYECTO = Path.cwd().parent
RUTA_MLFLOW = RUTA_PROYECTO / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{RUTA_MLFLOW.as_posix()}")
mlflow.set_experiment("deteccion_fraude")

with mlflow.start_run(run_name="xgboost_randomized_search"):

    mlflow.log_params(busqueda.best_params_)
    mlflow.log_param("metodo_optimizacion", "RandomizedSearchCV")
    mlflow.log_param("n_iter", 6)
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("tamano_muestra", 200000)
    mlflow.log_metric(
        "best_cv_average_precision",
        float(busqueda.best_score_)
    )
    mlflow.log_metric("tiempo_optimizacion_min", float(tiempo_opt))

    print("Optimización registrada en MLflow.")
    print("Run ID:", mlflow.active_run().info.run_id)

Optimización registrada en MLflow.
Run ID: 00d822bbde544e509742a6db57555b1d


In [11]:
# Entrenar el modelo con los mejores hiperparámetros
modelo_xgb_opt = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=4,
    **busqueda.best_params_
)

inicio = time.time()

modelo_xgb_opt.fit(
    X_train_transformed,
    y_train_np
)

tiempo_opt_final = (time.time() - inicio) / 60

print("Modelo optimizado entrenado correctamente.")
print(f"Tiempo: {tiempo_opt_final:.2f} minutos")

Modelo optimizado entrenado correctamente.
Tiempo: 1.28 minutos


In [12]:
import joblib
from pathlib import Path

RUTA_PROYECTO = Path.cwd().parent
RUTA_MODELO_OPT = RUTA_PROYECTO / "models" / "xgboost_fraude_optimizado.joblib"

joblib.dump(modelo_xgb_opt, RUTA_MODELO_OPT)

print("Modelo optimizado guardado correctamente.")
print(f"Ruta: {RUTA_MODELO_OPT}")

Modelo optimizado guardado correctamente.
Ruta: d:\ML_III\models\xgboost_fraude_optimizado.joblib
